# Stage 1 - Non-Instruction Fine-Tuning

**Project:** PrepMind - GenAI / Agentic AI Learning Assistant
**Base model:** `Qwen2.5-1.5B-Instruct` (loaded 4-bit via Unsloth)
**Goal:** Continue pre-training the base model on raw domain text (GenAI / Agentic AI explanations)
so it absorbs domain vocabulary, writing style, and background knowledge *before* instruction
tuning. This stage does **not** teach the model to follow instructions - it just adapts its
language modeling to the domain, the same way "domain-adaptive pretraining" works in industry.

Run this notebook top-to-bottom on a Google Colab **T4 GPU** runtime
(`Runtime -> Change runtime type -> T4 GPU`).


## 1. Install dependencies

In [ ]:
%%capture
import os, sys

# Unsloth + trl + peft + accelerate + bitsandbytes, pinned to versions known to work together on Colab
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets


## 2. Clone the project repo (datasets live here)

Replace the URL if you forked the repo.

In [ ]:
REPO_URL = "https://github.com/Abhishek15016/prepmind.git"
!git clone $REPO_URL project
%cd project


## 3. Load and inspect the raw domain text

`data/non_instruction_data.txt` contains 50+ paragraphs of raw GenAI / Agentic AI domain
explanations, separated by `## ` headings and `====` dividers.

In [ ]:
with open("data/non_instruction_data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split("=" * 40) if p.strip()]
print(f"Loaded {len(paragraphs)} raw domain paragraphs")
print(paragraphs[0][:500])


## 4. Clean and chunk the text

We chunk on paragraph boundaries (each paragraph is already a coherent, self-contained
chunk of domain knowledge) rather than a fixed token count, so the model never sees a
chunk that trails off mid-explanation.

In [ ]:
import re

def clean(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

chunks = [clean(p) for p in paragraphs if len(p) > 100]
print(f"{len(chunks)} usable chunks after cleaning")
print(f"Avg chunk length (chars): {sum(len(c) for c in chunks) / len(chunks):.0f}")


## 5. Build a Hugging Face Dataset for causal-LM continued pretraining

In [ ]:
from datasets import Dataset

raw_dataset = Dataset.from_dict({"text": chunks})
raw_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)
print(raw_dataset)


## 6. Load the base model with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None  # auto-detect (bf16 on Ampere+/T4 supports fp16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)


## 7. Apply LoRA adapters

For non-instruction (continued pretraining) fine-tuning we target the same attention +
MLP projection layers as instruction tuning, but use a slightly lower rank since the goal
is light domain adaptation, not teaching a new skill.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()


## 8. Tokenize for causal language modeling (packed sequences)

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=max_seq_length)

tokenized = raw_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])


## 9. Train (non-instruction / continued pretraining)

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="outputs/non_instruction",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=2,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    dataset_text_field=None,
    data_collator=data_collator,
    args=training_args,
)

trainer_stats = trainer.train()


## 10. Save the Stage-1 (non-instruction) adapter

In [ ]:
model.save_pretrained("outputs/non_instruction_adapter")
tokenizer.save_pretrained("outputs/non_instruction_adapter")
print("Saved LoRA adapter to outputs/non_instruction_adapter")

# Optional: push to Hugging Face Hub
# from huggingface_hub import login
# login(token="hf_xxx")
# model.push_to_hub("your-username/prepmind-non-instruction-adapter")


## 11. Quick sanity test

We are *not* expecting instruction-following yet - this stage only checks that the model
now completes domain text more fluently / with more domain vocabulary than the raw base
model would.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "In Retrieval-Augmented Generation, the retriever's job is to"
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=120, do_sample=True, temperature=0.7)
print(tokenizer.decode(output[0], skip_special_tokens=True))


## Notes

- This stage deliberately uses a *small* LoRA rank (16) and *few* epochs (2) - the goal is
  gentle domain adaptation, not memorization. Watch the eval loss; if it drops far below
  train loss or the model starts regurgitating chunks verbatim, reduce epochs or rank.
- Stage 2 (`instruction_finetuning.ipynb`) loads this saved adapter and continues training
  on the instruction dataset.
